## 1. An Overview Kernel Gradient Discrepancy (KGD)

KGD provides a scalar diagnostic of how close an empirical particle swarm, $Q_n$, is to a target distribution. It is the RKHS norm of the projected Wasserstein gradient field,
$$KGD_Q= ||b(Q,\cdot)||_{\mathcal{H}_K(Q)},$$
where $b(Q,x_i)$ is the variational gradient field, in other words the net deterministic force vector acting on particle. In our case this is represented as a combination of our Wasserstein loss gradient, our regularisation pull to the prior, and the addtion of some brownian noise:
$$d\vartheta^{(j)}_t = -\left\{ \lambda_n \nabla_\vartheta \mathcal{L}(Q_t)[\vartheta^{(j)}_t] - \nabla_\vartheta \log \pi(\vartheta^{(j)}_t) \right\} dt + \sqrt{2}dB^{(j)}_t$$

The empirical equivalent is as seen below:
$$\vartheta_{t+1}^{\vartheta (j)} = \vartheta_t^{\vartheta (j)} + \epsilon_t\underbrace{\left(-\lambda_n \, \nabla_{\vartheta}\mathcal{L}(Q_n^{(t)})(\vartheta_t^{(j)}) + \nabla_{\vartheta}\log\pi\big(\vartheta_t^{(j)}\big)\right)}_{b(Q_n^{(t)}, \vartheta^{(j)}_t)}\ + \sqrt{2\epsilon_t}\eta_t^{(j)}$$

Expanding the RKHS norm gives

$$\mathrm{KGD}^2(Q)=\iint b(Q,\vartheta)^T K(\vartheta,\vartheta') b(Q,\vartheta')\, dQ(\vartheta)\, dQ(\vartheta')$$
which, for an empirical particle measure $Q_n=\frac{1}{n}\sum_{i=1}^n \delta_{\vartheta^{(j)}},$ reduces to the computable estimator  
$$\mathrm{KGD}^2(Q_n)=\frac{1}{N_p^2}\sum_{j=1}^{N_p}\sum_{l=1}^{N_p} b(Q_n,\vartheta^{(j)})^{T}K(\vartheta^{(j)},\vartheta^{(l)})b(Q_n,\vartheta^{(l)})$$



## 2. Bridging the Notation

Before we can use KGD as a diagnostic tool, we must align the notation across our 2 pieces of core literature as the PrO posteriors paper (McLatchie et al., 2025) and the KGD paper (Chazal et al., 2026) use slightly different conventions.

| Concept / Variable | KGD Paper (Chazal et al.) | PrO Posteriors Paper (McLatchie et al.) | `pymc-prop` Code Base |
| :--- | :--- | :--- | :--- |
| **Domain Variable** | $x \in \mathbb{R}^d$ | $\vartheta \in \Theta$ | Unconstrained arrays (`value_vars` coordinates via `PointMapper`) |
| **Data Observations** | Unspecified / Implicit | $x_{1:n} = (x_1, \dots, x_n)$ | PyMC observed data array `x_obs` |
| **Particle Count** | $n$ | $N_p$ or $p$ | `particles` shape `(n_particles, d)` |
| **Data Fit Loss** | $L(Q)$ | $\frac{1}{n} \sum_{i=1}^n S(P_Q, x_i)$ | `compile_drift_for_logscore` / `ScoringRule` protocol |
| **Loss Scaling / Temp** | Unscaled ($\lambda = 1$) | Learning rate $\lambda_n$ | `learning_rate` parameter (default `1.0`) |
| **Target Objective** | $J(Q) = L(Q) + D_{\text{KL}}(Q \parallel Q_0)$ | $J(Q) = \lambda_n L(Q) + D_{\text{KL}}(Q \parallel \Pi)$ | Calculated via interacting drift + prior pull |
| **Data Gradient Force** | $\nabla \delta L(Q)[x]$ | $-\lambda_n \nabla_\vartheta \mathcal{L}(Q)[\vartheta^{(j)}]$ | `compiled_drift(particles)` output tensor `(N_p, d)` |
| **Prior Anchor Force** | $\nabla \log q_0(x)$ | $\nabla_\vartheta \log \pi(\vartheta^{(j)})$ | `compile_prior_gradient(model)` |
| **Particle Interactions** | Implicit in variational gradient $b(Q,x)$ | Pairwise mixture weights $w_{i,j}$ / MMD gradients | Evaluated elementwise across observations in unconstrained space |
| **Convergence Metric** | $\mathrm{KGD}^2(Q_n) = \frac{1}{n^2}\sum_{i,j} b_i^T K_{i,j} b_j$ | Monitored via WGF stationarity | This notebook's addition |

## 3. Implementing KGD

The beauty of evaluating KGD within `pymc-prop` is that the most computationally expensive operations, evaluating the log-score gradients across the dataset and the prior, are already handled natively inside the Wasserstein Gradient Flow (WGF) loop.

$b(Q, \vartheta)$ represents the variational gradient (the net deterministic force) which is currently calculated by the `time_step` function, when isolating the variational gradient vector $b(Q, \vartheta)$ from this time_step() update, we must extract only the net deterministic drift, the combination of the prior pull and the data-fit score scaled by $\lambda_n$. We explicitly exclude both the numerical integration step_size and the injected Brownian noise. Excluding step_size ensures KGD remains a continuous-time metric that serves as an absolute, step-size-independent stopping criterion, while omitting Brownian noise prevents stochastic variance from obscuring the smooth convergence of our diagnostic to zero.

With the pure drift tensor (shape `N_p, d`) isolated, computing KGD requires only three steps:
1. Extract the raw `drift` and `particles` tensors at a given time step.
2. Define a positive-definite RKHS kernel to measure the spatial correlation of the particles.
3. Compute the double sum estimator of the kernelized forces efficiently.